In [6]:
# Import libraries

import requests
import pandas as pd

# Set the pedestrian API URL

pedestrian_url = (
    "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
    "pedestrian-counting-system-monthly-counts-per-hour/records"
)

In [7]:
# Get the latest available date

params = {
    "select": "max(sensing_date) as latest_date",
    "limit": 1
}

response = requests.get(pedestrian_url, params=params)
response.raise_for_status()

latest_data = response.json()
end_date = pd.to_datetime(latest_data["results"][0]["latest_date"])

print("Latest date:", end_date.date())

Latest date: 2026-09-16


In [8]:
# Set the download range to the last 3 months

start_date = end_date - pd.DateOffset(months=3)

print("Start date:", start_date.date())
print("End date:", end_date.date())

Start date: 2026-06-16
End date: 2026-09-16


In [9]:
# Download pedestrian data day by day

all_data = []
current_date = start_date

while current_date <= end_date:
    next_date = current_date + pd.Timedelta(days=1)
    offset = 0

    while True:
        params = {
            "where": (
                f"sensing_date >= '{current_date.date()}' "
                f"AND sensing_date < '{next_date.date()}'"
            ),
            "order_by": "id",
            "limit": 100,
            "offset": offset
        }

        response = requests.get(pedestrian_url, params=params)
        response.raise_for_status()
        data = response.json()

        if data["total_count"] > 10000:
            raise ValueError("Too many records in one day. Use a smaller range.")

        results = data["results"]
        all_data.extend(results)

        if offset + len(results) >= data["total_count"]:
            break

        if len(results) == 0:
            raise ValueError("Download stopped before all records were received.")

        offset += 100

    current_date = next_date

print("Downloaded rows:", len(all_data))

Downloaded rows: 206234


In [10]:
# Create the raw data table

pedestrian_raw = pd.DataFrame(all_data)

# Make a copy for cleaning

pedestrian_clean = pedestrian_raw.copy()

print("Rows and columns:", pedestrian_raw.shape)
pedestrian_raw.head()

Rows and columns: (206234, 9)


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,2020260616,2,2026-06-16,0,13,8,21,Bou283_T,"{'lon': 144.96516718, 'lat': -37.81380668}"
1,2120260616,2,2026-06-16,1,4,7,11,Bou283_T,"{'lon': 144.96516718, 'lat': -37.81380668}"
2,2220260616,2,2026-06-16,2,0,4,4,Bou283_T,"{'lon': 144.96516718, 'lat': -37.81380668}"
3,2320260616,2,2026-06-16,3,4,9,13,Bou283_T,"{'lon': 144.96516718, 'lat': -37.81380668}"
4,2420260616,2,2026-06-16,4,6,2,8,Bou283_T,"{'lon': 144.96516718, 'lat': -37.81380668}"


In [11]:
# Check data types and missing values

print("Data types:")
print(pedestrian_clean.dtypes)

print("\nMissing values:")
print(pedestrian_clean.isna().sum())

Data types:
id                  int64
location_id         int64
sensing_date       object
hourday             int64
direction_1         int64
direction_2         int64
pedestriancount     int64
sensor_name        object
location           object
dtype: object

Missing values:
id                 0
location_id        0
sensing_date       0
hourday            0
direction_1        0
direction_2        0
pedestriancount    0
sensor_name        0
location           0
dtype: int64


In [12]:
# Convert dates and check for invalid values

pedestrian_clean["sensing_date"] = pd.to_datetime(
    pedestrian_clean["sensing_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("Invalid or missing dates:",
      pedestrian_clean["sensing_date"].isna().sum())
print("Start date:", pedestrian_clean["sensing_date"].min())
print("End date:", pedestrian_clean["sensing_date"].max())

Invalid or missing dates: 0
Start date: 2026-06-16 00:00:00
End date: 2026-09-16 00:00:00


In [13]:
# Convert pedestrian numeric fields

number_columns = [
    "location_id", "hourday",
    "direction_1", "direction_2", "pedestriancount"
]

for column in number_columns:
    pedestrian_clean[column] = pd.to_numeric(
        pedestrian_clean[column], errors="coerce"
    )

# Flag values that are not whole numbers

invalid_numbers = (
    pedestrian_clean[number_columns].notna() &
    (pedestrian_clean[number_columns] % 1 != 0)
).any(axis=1)

print("Rows with non-integer values:", invalid_numbers.sum())

Rows with non-integer values: 0


In [14]:
# Check duplicate location-date-hour records

duplicate_rows = pedestrian_clean.duplicated(
    subset=["location_id", "sensing_date", "hourday"]
)

print("Duplicate records:", duplicate_rows.sum())

Duplicate records: 0


In [15]:
# Check hours and pedestrian counts

invalid_hours = ~pedestrian_clean["hourday"].between(0, 23)

negative_counts = (
    pedestrian_clean[["direction_1", "direction_2", "pedestriancount"]] < 0
).any(axis=1)

different_totals = (
    pedestrian_clean["direction_1"] + pedestrian_clean["direction_2"]
    != pedestrian_clean["pedestriancount"]
)

print("Invalid hours:", invalid_hours.sum())
print("Rows with negative counts:", negative_counts.sum())
print("Rows with different totals:", different_totals.sum())

Invalid hours: 0
Rows with negative counts: 0
Rows with different totals: 0


In [16]:
# Split and check pedestrian coordinates

coordinates = pd.json_normalize(
    pedestrian_clean["location"]
).reindex(columns=["lat", "lon"])

coordinates.index = pedestrian_clean.index

pedestrian_clean["latitude"] = pd.to_numeric(
    coordinates["lat"], errors="coerce"
)
pedestrian_clean["longitude"] = pd.to_numeric(
    coordinates["lon"], errors="coerce"
)

invalid_pedestrian_coordinates = (
    ~pedestrian_clean["latitude"].between(-90, 90) |
    ~pedestrian_clean["longitude"].between(-180, 180)
)

print("Rows with invalid or missing coordinates:",
      invalid_pedestrian_coordinates.sum())

Rows with invalid or missing coordinates: 0


In [17]:
# Separate records that need review

missing_values = pedestrian_clean[
    ["location_id", "sensing_date", "hourday",
     "direction_1", "direction_2", "pedestriancount"]
].isna().any(axis=1)

# Flag all records with duplicate location-date-hour keys

duplicate_records = pedestrian_clean.duplicated(
    subset=["location_id", "sensing_date", "hourday"],
    keep=False
)

needs_review = (
    missing_values | invalid_hours | negative_counts |
    different_totals | duplicate_records | invalid_numbers |
    invalid_pedestrian_coordinates
)

pedestrian_review = pedestrian_clean[needs_review].copy()
pedestrian_clean = pedestrian_clean[~needs_review].copy()

print("Rows for review:", len(pedestrian_review))
print("Clean rows:", len(pedestrian_clean))

Rows for review: 0
Clean rows: 206234


In [18]:
# Check hourly coverage for each location and date

hours_per_day = pedestrian_clean.groupby(
    ["location_id", "sensing_date"]
)["hourday"].nunique()

print("Location-day pairs:", len(hours_per_day))
print("Pairs with fewer than 24 hours:", (hours_per_day < 24).sum())

Location-day pairs: 9192
Pairs with fewer than 24 hours: 3110


In [19]:
# Sort the cleaned records

pedestrian_clean = pedestrian_clean.sort_values(
    ["location_id", "sensing_date", "hourday"]
).reset_index(drop=True)

pedestrian_clean.head()

,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location,latitude,longitude
0,1920260630,1,2026-06-30,9,90,148,238,Bou292_T,"{'lon': 144.96515323, 'lat': -37.81349441}",-37.813494,144.965153
1,11020260630,1,2026-06-30,10,312,414,726,Bou292_T,"{'lon': 144.96515323, 'lat': -37.81349441}",-37.813494,144.965153
2,11120260630,1,2026-06-30,11,586,576,1162,Bou292_T,"{'lon': 144.96515323, 'lat': -37.81349441}",-37.813494,144.965153
3,11220260630,1,2026-06-30,12,1141,999,2140,Bou292_T,"{'lon': 144.96515323, 'lat': -37.81349441}",-37.813494,144.965153
4,11320260630,1,2026-06-30,13,706,890,1596,Bou292_T,"{'lon': 144.96515323, 'lat': -37.81349441}",-37.813494,144.965153


In [20]:
# Summarise the cleaning results

print("Raw rows:", len(pedestrian_raw))
print("Clean rows:", len(pedestrian_clean))
print("Rows for review:", len(pedestrian_review))

print("Start date:", pedestrian_clean["sensing_date"].min())
print("End date:", pedestrian_clean["sensing_date"].max())

print("Rows accounted for:",
      len(pedestrian_raw) == len(pedestrian_clean) + len(pedestrian_review))

Raw rows: 206234
Clean rows: 206234
Rows for review: 0
Start date: 2026-06-16 00:00:00
End date: 2026-09-16 00:00:00
Rows accounted for: True


In [21]:
# Stop if no cleaned pedestrian records remain

if pedestrian_clean.empty:
    raise ValueError("No cleaned pedestrian records. CSV was not updated.")

In [22]:
# Export the selected columns

pedestrian_clean[
    ["location_id", "sensing_date", "hourday", "pedestriancount",
     "sensor_name", "latitude", "longitude"]
].to_csv("pedestrian_clean.csv", index=False)

In [23]:
# Set the sensor locations API URL

sensor_url = (
    "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
    "pedestrian-counting-system-sensor-locations/records"
)

# Download sensor locations

all_sensors = []
offset = 0

while True:
    params = {
        "limit": 100,
        "offset": offset,
        "order_by": "location_id"
    }

    response = requests.get(sensor_url, params=params)
    response.raise_for_status()
    data = response.json()

    if data["total_count"] > 10000:
        raise ValueError("Too many records for this pagination method.")

    results = data["results"]
    all_sensors.extend(results)

    if offset + len(results) >= data["total_count"]:
        break

    if len(results) == 0:
        raise ValueError("Download stopped before all records were received.")

    offset += 100

# Create raw and working tables

sensors_raw = pd.DataFrame(all_sensors)
sensors_clean = sensors_raw.copy()

print("Downloaded rows:", len(sensors_raw))
sensors_raw.head()

Downloaded rows: 134


,location_id,sensor_description,sensor_name,installation_date,note,location_type,status,direction_1,direction_2,latitude,longitude,location
0,1,Bourke Street Mall (North),Bou292_T,2009-03-24,None,Outdoor,A,East,West,-37.813494,144.965153,"{'lon': 144.96515323, 'lat': -37.81349441}"
1,2,Bourke Street Mall (South),Bou283_T,2009-03-30,None,Outdoor,A,East,West,-37.813807,144.965167,"{'lon': 144.96516718, 'lat': -37.81380668}"
2,3,Melbourne Central,Swa295_T,2009-03-25,None,Outdoor,A,North,South,-37.811015,144.964295,"{'lon': 144.96429485, 'lat': -37.81101524}"
3,4,Town Hall (West),Swa123_T,2009-03-23,None,Outdoor,A,North,South,-37.814880,144.966088,"{'lon': 144.9660878, 'lat': -37.81487988}"
4,5,Princes Bridge,PriNW_T,2009-03-26,Replace with: 00:6e:02:01:9e:54,Outdoor,A,North,South,-37.818742,144.967877,"{'lon': 144.96787656, 'lat': -37.81874249}"


In [24]:
# Check data types and missing values

print("Data types:")
print(sensors_clean.dtypes)

print("\nMissing values:")
print(sensors_clean.isna().sum())

Data types:
location_id             int64
sensor_description     object
sensor_name            object
installation_date      object
note                   object
location_type          object
status                 object
direction_1            object
direction_2            object
latitude              float64
longitude             float64
location               object
dtype: object

Missing values:
location_id            0
sensor_description     0
sensor_name            0
installation_date      3
note                  96
location_type          0
status                 0
direction_1           34
direction_2           34
latitude               0
longitude              0
location               0
dtype: int64


In [25]:
# Convert installation dates

sensors_clean["installation_date"] = pd.to_datetime(
    sensors_raw["installation_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

invalid_dates = (
    sensors_raw["installation_date"].notna() &
    sensors_clean["installation_date"].isna()
)

print("Invalid installation dates:", invalid_dates.sum())

Invalid installation dates: 0


In [26]:
# Convert location IDs to numbers

sensors_clean["location_id"] = pd.to_numeric(
    sensors_clean["location_id"], errors="coerce"
)

invalid_sensor_ids = (
    sensors_clean["location_id"].notna() &
    (sensors_clean["location_id"] % 1 != 0)
)

print("Non-integer location IDs:", invalid_sensor_ids.sum())

Non-integer location IDs: 0


In [27]:
# Convert coordinates to numbers

for column in ["latitude", "longitude"]:
    sensors_clean[column] = pd.to_numeric(
        sensors_clean[column], errors="coerce"
    )

# Check duplicate IDs and coordinates

duplicate_sensors = sensors_clean.duplicated(
    subset=["location_id"], keep=False
)

invalid_coordinates = (
    ~sensors_clean["latitude"].between(-90, 90) |
    ~sensors_clean["longitude"].between(-180, 180)
)

print("Rows with duplicate IDs:", duplicate_sensors.sum())
print("Rows with invalid or missing coordinates:", invalid_coordinates.sum())

Rows with duplicate IDs: 0
Rows with invalid or missing coordinates: 0


In [28]:
# Clean text fields

for column in ["sensor_description", "sensor_name", "location_type", "status"]:
    sensors_clean[column] = sensors_clean[column].astype("string").str.strip()
    sensors_clean[column] = sensors_clean[column].replace("", pd.NA)

sensors_clean["status"] = sensors_clean["status"].str.upper()

print(sensors_clean[
    ["sensor_description", "sensor_name", "location_type", "status"]
].isna().sum())

print(sensors_clean["status"].value_counts(dropna=False))

sensor_description    0
sensor_name           0
location_type         0
status                0
dtype: int64
status
A    134
Name: count, dtype: Int64


In [29]:
# Separate sensor records that need review

missing_sensor_values = sensors_clean[
    ["location_id", "sensor_description", "sensor_name",
     "latitude", "longitude", "status"]
].isna().any(axis=1)

sensor_needs_review = (
    missing_sensor_values | duplicate_sensors |
    invalid_coordinates | invalid_dates | invalid_sensor_ids
)

sensors_review = sensors_clean[sensor_needs_review].copy()
sensors_clean = sensors_clean[~sensor_needs_review].copy()

print("Rows for review:", len(sensors_review))
print("Clean rows:", len(sensors_clean))

Rows for review: 0
Clean rows: 134


In [30]:
# Sort sensor records

sensors_clean = sensors_clean.sort_values(
    "location_id"
).reset_index(drop=True)

# Stop if no cleaned sensor records remain

if sensors_clean.empty:
    raise ValueError("No cleaned sensor records. CSV was not updated.")

# Export the selected columns

sensors_clean[
    ["location_id", "sensor_description", "sensor_name",
     "installation_date", "note", "location_type", "status",
     "latitude", "longitude"]
].to_csv("sensor_locations_clean.csv", index=False)

In [31]:
# Set the microclimate API URL

noise_url = (
    "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
    "microclimate-sensors-data/records"
)

# Set the download date range

noise_start = pd.Timestamp(start_date.date())
noise_end = pd.Timestamp(end_date.date()) + pd.Timedelta(days=1)

noise_start = noise_start.tz_localize("Australia/Melbourne")
noise_end = noise_end.tz_localize("Australia/Melbourne")

print("Start:", noise_start)
print("End (not included):", noise_end)

Start: 2026-06-16 00:00:00+10:00
End (not included): 2026-09-17 00:00:00+10:00


In [32]:
# Download microclimate data day by day

all_noise = []
current_date = noise_start

while current_date < noise_end:
    next_date = current_date + pd.DateOffset(days=1)
    offset = 0

    while True:
        params = {
            "where": (
                f"received_at >= '{current_date.isoformat()}' "
                f"AND received_at < '{next_date.isoformat()}'"
            ),
            "order_by": "received_at, device_id",
            "limit": 100,
            "offset": offset
        }

        response = requests.get(noise_url, params=params)
        response.raise_for_status()
        data = response.json()

        if data["total_count"] > 10000:
            raise ValueError("Too many records in one day. Use a smaller range.")

        results = data["results"]
        all_noise.extend(results)

        if offset + len(results) >= data["total_count"]:
            break

        if len(results) == 0:
            raise ValueError("Download stopped before all records were received.")

        offset += 100

    current_date = next_date

# Create raw and working tables

noise_raw = pd.DataFrame(all_noise)
noise_clean = noise_raw.copy()

print("Downloaded rows:", len(noise_raw))
noise_raw.head()

Downloaded rows: 69461


,device_id,received_at,sensorlocation,latlong,minimumwinddirection,averagewinddirection,maximumwinddirection,minimumwindspeed,averagewindspeed,gustwindspeed,airtemperature,relativehumidity,atmosphericpressure,pm25,pm10,noise
0,ICTMicroclimate-06,2026-06-15T14:00:43+00:00,Tram Stop 7B – Melbourne Tennis Centre Precinc...,"{'lon': 144.9787211, 'lat': -37.8194993}",0.0,72.0,353.0,0.0,0.9,2.3,13.9,86.000000,1023.3,8.0,11.0,74.400000
1,ICTMicroclimate-10,2026-06-15T14:06:39+00:00,1 Treasury Place,"{'lon': 144.9745395, 'lat': -37.8128595}",1.0,319.0,356.0,0.9,1.5,2.6,14.1,95.800000,1019.4,10.0,15.0,62.300000
2,ICTMicroclimate-05,2026-06-15T14:08:47+00:00,Enterprize Park - Pole ID: COM1667,"{'lon': 144.9591192, 'lat': -37.8204083}",0.0,4.0,352.0,0.0,0.3,1.1,15.2,92.900000,1022.9,8.0,11.0,67.800000
3,ICTMicroclimate-01,2026-06-15T14:12:27+00:00,Birrarung Marr Park - Pole 1131,"{'lon': 144.9716404, 'lat': -37.8185931}",NaN,128.0,NaN,NaN,0.5,NaN,12.3,94.800003,1022.0,10.0,15.0,49.099998
4,aws5-0999,2026-06-15T14:12:47+00:00,Royal Park Asset ID: COM2707,"{'lon': 144.9519007, 'lat': -37.7956167}",0.0,0.0,0.0,0.0,0.0,0.0,11.6,97.800000,1020.8,NaN,NaN,NaN


In [33]:
# Select fields needed for noise data cleaning

noise_clean = noise_raw[
    ["device_id", "received_at", "sensorlocation", "latlong", "noise"]
].copy()

# Check data types and missing values

print("Data types:")
print(noise_clean.dtypes)

print("\nMissing values:")
print(noise_clean.isna().sum())

Data types:
device_id          object
received_at        object
sensorlocation     object
latlong            object
noise             float64
dtype: object

Missing values:
device_id            0
received_at          0
sensorlocation       0
latlong              0
noise             7574
dtype: int64


In [34]:
# Convert timestamps to Melbourne time

noise_clean["received_at"] = pd.to_datetime(
    noise_clean["received_at"],
    utc=True,
    errors="coerce"
)

noise_clean["received_at"] = noise_clean["received_at"].dt.tz_convert(
    "Australia/Melbourne"
)

print("Invalid or missing timestamps:",
      noise_clean["received_at"].isna().sum())
print("Start:", noise_clean["received_at"].min())
print("End:", noise_clean["received_at"].max())

Invalid or missing timestamps: 0
Start: 2026-06-16 00:00:43+10:00
End: 2026-09-16 23:56:36+10:00


In [35]:
# Split coordinates into latitude and longitude
coordinates = pd.json_normalize(
    noise_clean["latlong"]
).reindex(columns=["lat", "lon"])

coordinates.index = noise_clean.index

noise_clean["latitude"] = pd.to_numeric(
    coordinates["lat"], errors="coerce"
)
noise_clean["longitude"] = pd.to_numeric(
    coordinates["lon"], errors="coerce"
)

# Check coordinate ranges

invalid_noise_coordinates = (
    ~noise_clean["latitude"].between(-90, 90) |
    ~noise_clean["longitude"].between(-180, 180)
)

print("Rows with invalid or missing coordinates:",
      invalid_noise_coordinates.sum())

Rows with invalid or missing coordinates: 0


In [36]:
# Clean device IDs and location names

for column in ["device_id", "sensorlocation"]:
    noise_clean[column] = noise_clean[column].astype("string").str.strip()
    noise_clean[column] = noise_clean[column].replace("", pd.NA)

# Convert noise readings to numbers

noise_clean["noise"] = pd.to_numeric(
    noise_clean["noise"], errors="coerce"
)

print("Missing values after conversion:")
print(noise_clean[["device_id", "sensorlocation", "noise"]].isna().sum())

Missing values after conversion:
device_id            0
sensorlocation       0
noise             7574
dtype: int64


In [37]:
# Check duplicate device-timestamp records

duplicate_noise = noise_clean.duplicated(
    subset=["device_id", "received_at"],
    keep=False
)

print("Rows with duplicate timestamps:", duplicate_noise.sum())

noise_clean.loc[
    duplicate_noise,
    ["device_id", "received_at", "noise"]
].sort_values(["device_id", "received_at"]).head(10)

Rows with duplicate timestamps: 846


,device_id,received_at,noise
2350,ICTMicroclimate-01,2026-06-18 18:33:30+10:00,51.799999
2351,ICTMicroclimate-01,2026-06-18 18:33:30+10:00,55.200001
6341,ICTMicroclimate-01,2026-06-23 11:22:40+10:00,54.099998
6342,ICTMicroclimate-01,2026-06-23 11:22:40+10:00,52.700001
6659,ICTMicroclimate-01,2026-06-23 20:22:52+10:00,49.500000
6660,ICTMicroclimate-01,2026-06-23 20:22:52+10:00,48.099998
7414,ICTMicroclimate-01,2026-06-24 17:38:32+10:00,54.200001
7415,ICTMicroclimate-01,2026-06-24 17:38:32+10:00,52.599998
10614,ICTMicroclimate-01,2026-06-28 11:47:26+10:00,52.000000
10615,ICTMicroclimate-01,2026-06-28 11:47:26+10:00,50.900002


In [38]:
# Check noise values

print(noise_clean["noise"].describe())
print("Zero readings:", (noise_clean["noise"] == 0).sum())
print("Negative readings:", (noise_clean["noise"] < 0).sum())

count    61887.000000
mean        63.349505
std         13.329180
min          0.000000
25%         56.500000
50%         64.400000
75%         69.900000
max        127.600000
Name: noise, dtype: float64
Zero readings: 437
Negative readings: 0


In [39]:
# Flag zero noise readings

noise_clean["zero_noise"] = noise_clean["noise"] == 0

# Separate records that need review

missing_noise_values = noise_clean[
    ["device_id", "received_at", "noise", "latitude", "longitude"]
].isna().any(axis=1)

noise_needs_review = (
    missing_noise_values |
    invalid_noise_coordinates |
    duplicate_noise
)

noise_review = noise_clean[noise_needs_review].copy()
noise_clean = noise_clean[~noise_needs_review].copy()

print("Rows for review:", len(noise_review))
print("Clean rows:", len(noise_clean))
print("Zero readings retained:", noise_clean["zero_noise"].sum())

Rows for review: 8420
Clean rows: 61041
Zero readings retained: 425


In [40]:
# Sort the cleaned noise records

noise_clean = noise_clean.sort_values(
    ["device_id", "received_at"]
).reset_index(drop=True)

# Check row totals

print("Rows accounted for:",
      len(noise_raw) == len(noise_clean) + len(noise_review))

# Stop if no cleaned noise records remain

if noise_clean.empty:
    raise ValueError("No cleaned noise records. CSV was not updated.")

# Export the selected columns

noise_clean[
    ["device_id", "received_at", "sensorlocation",
     "latitude", "longitude", "noise", "zero_noise"]
].to_csv("noise_clean.csv", index=False)

Rows accounted for: True


In [41]:
# Prepare pedestrian data for joining

pedestrian_join = pedestrian_clean[
    ["location_id", "sensing_date", "hourday",
     "pedestriancount", "sensor_name",
     "latitude", "longitude"]
].copy()

# Keep daytime hours only

pedestrian_join = pedestrian_join[
    pedestrian_join["hourday"].between(9, 17)
].copy()

print("Pedestrian rows for joining:", len(pedestrian_join))
pedestrian_join.head()

Pedestrian rows for joining: 81366


,location_id,sensing_date,hourday,pedestriancount,sensor_name,latitude,longitude
0,1,2026-06-30,9,238,Bou292_T,-37.813494,144.965153
1,1,2026-06-30,10,726,Bou292_T,-37.813494,144.965153
2,1,2026-06-30,11,1162,Bou292_T,-37.813494,144.965153
3,1,2026-06-30,12,2140,Bou292_T,-37.813494,144.965153
4,1,2026-06-30,13,1596,Bou292_T,-37.813494,144.965153


In [42]:
# Create date and hour columns for noise data

noise_clean["sensing_date"] = noise_clean["received_at"].dt.date
noise_clean["hourday"] = noise_clean["received_at"].dt.hour

# Convert pedestrian dates to date only

pedestrian_join["sensing_date"] = (
    pedestrian_join["sensing_date"].dt.date
)

# Keep daytime noise records only

noise_daytime = noise_clean[
    noise_clean["hourday"].between(9, 17)
].copy()

print("Daytime noise rows:", len(noise_daytime))
noise_daytime.head()

Daytime noise rows: 22929


,device_id,received_at,sensorlocation,latlong,noise,latitude,longitude,zero_noise,sensing_date,hourday
36,ICTMicroclimate-01,2026-06-16 09:12:49+10:00,Birrarung Marr Park - Pole 1131,"{'lon': 144.9716404, 'lat': -37.8185931}",55.599998,-37.818593,144.97164,False,2026-06-16,9
37,ICTMicroclimate-01,2026-06-16 09:27:48+10:00,Birrarung Marr Park - Pole 1131,"{'lon': 144.9716404, 'lat': -37.8185931}",54.700001,-37.818593,144.97164,False,2026-06-16,9
38,ICTMicroclimate-01,2026-06-16 09:42:48+10:00,Birrarung Marr Park - Pole 1131,"{'lon': 144.9716404, 'lat': -37.8185931}",52.299999,-37.818593,144.97164,False,2026-06-16,9
39,ICTMicroclimate-01,2026-06-16 09:57:51+10:00,Birrarung Marr Park - Pole 1131,"{'lon': 144.9716404, 'lat': -37.8185931}",55.299999,-37.818593,144.97164,False,2026-06-16,9
40,ICTMicroclimate-01,2026-06-16 10:12:51+10:00,Birrarung Marr Park - Pole 1131,"{'lon': 144.9716404, 'lat': -37.8185931}",55.700001,-37.818593,144.97164,False,2026-06-16,10


In [43]:
# Calculate average noise for each sensor and hour

noise_hourly = noise_daytime.groupby(
    ["device_id", "sensorlocation",
     "latitude", "longitude",
     "sensing_date", "hourday"],
    as_index=False
)["noise"].mean()

noise_hourly = noise_hourly.rename(
    columns={"noise": "average_noise"}
)

print("Hourly noise rows:", len(noise_hourly))
noise_hourly.head()

Hourly noise rows: 5830


,device_id,sensorlocation,latitude,longitude,sensing_date,hourday,average_noise
0,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,-37.818593,144.97164,2026-06-16,9,54.474999
1,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,-37.818593,144.97164,2026-06-16,10,53.375000
2,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,-37.818593,144.97164,2026-06-16,11,53.774999
3,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,-37.818593,144.97164,2026-06-16,12,53.599999
4,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,-37.818593,144.97164,2026-06-16,13,56.225001


In [44]:
# Get unique pedestrian sensor locations

pedestrian_locations = pedestrian_join[
    ["location_id", "sensor_name",
     "latitude", "longitude"]
].drop_duplicates(
    subset=["location_id"]
).reset_index(drop=True)

# Get unique noise sensor locations

noise_locations = noise_hourly[
    ["device_id", "sensorlocation",
     "latitude", "longitude"]
].drop_duplicates(
    subset=["device_id"]
).reset_index(drop=True)

print("Pedestrian sensors:", len(pedestrian_locations))
print("Noise sensors:", len(noise_locations))

Pedestrian sensors: 100
Noise sensors: 11


In [45]:
# Import numpy for distance calculation

import numpy as np

In [46]:
# Calculate distance between two coordinates

def calculate_distance(lat1, lon1, lat2, lon2):

    earth_radius = 6371000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    lat_difference = lat2 - lat1
    lon_difference = lon2 - lon1

    a = (
        np.sin(lat_difference / 2) ** 2 +
        np.cos(lat1) * np.cos(lat2) *
        np.sin(lon_difference / 2) ** 2
    )

    distance = 2 * earth_radius * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

    return distance

In [47]:
# Match each pedestrian sensor with the nearest noise sensor

sensor_matches = []

for _, pedestrian_sensor in pedestrian_locations.iterrows():

    noise_temp = noise_locations.copy()

    noise_temp["distance_m"] = calculate_distance(
        pedestrian_sensor["latitude"],
        pedestrian_sensor["longitude"],
        noise_temp["latitude"],
        noise_temp["longitude"]
    )

    nearest_noise = noise_temp.loc[
        noise_temp["distance_m"].idxmin()
    ]

    sensor_matches.append({
        "location_id": pedestrian_sensor["location_id"],
        "device_id": nearest_noise["device_id"],
        "noise_location": nearest_noise["sensorlocation"],
        "distance_m": nearest_noise["distance_m"]
    })

sensor_matches = pd.DataFrame(sensor_matches)

print("Matched sensors:", len(sensor_matches))
sensor_matches.head()

Matched sensors: 100


,location_id,device_id,noise_location,distance_m
0,1,ICTMicroclimate-03,CH1 rooftop,196.251171
1,2,ICTMicroclimate-03,CH1 rooftop,187.325311
2,3,ICTMicroclimate-03,CH1 rooftop,426.029960
3,4,ICTMicroclimate-03,CH1 rooftop,140.705296
4,5,ICTMicroclimate-08,Swanston St - Tram Stop 13 adjacent Federation...,32.457835


In [48]:
# Check sensor matching distances

print(sensor_matches["distance_m"].describe())

sensor_matches.sort_values(
    "distance_m",
    ascending=False
).head(10)

count     100.000000
mean      706.978444
std       717.595096
min        30.083907
25%       209.392114
50%       450.938100
75%       980.585179
max      3685.511162
Name: distance_m, dtype: float64


,location_id,device_id,noise_location,distance_m
61,85,ICTMicroclimate-09,SkyFarm (Jeff’s Shed). Rooftop - Melbourne Con...,3685.511162
57,76,ICTMicroclimate-09,SkyFarm (Jeff’s Shed). Rooftop - Melbourne Con...,3636.059416
92,180,ICTMicroclimate-09,SkyFarm (Jeff’s Shed). Rooftop - Melbourne Con...,3383.940320
62,86,ICTMicroclimate-03,CH1 rooftop,2008.652869
32,44,ICTMicroclimate-03,CH1 rooftop,1912.243845
63,87,ICTMicroclimate-03,CH1 rooftop,1905.263824
53,70,ICTMicroclimate-03,CH1 rooftop,1886.211587
38,50,ICTMicroclimate-10,1 Treasury Place,1764.854766
31,43,ICTMicroclimate-03,CH1 rooftop,1755.597698
30,42,ICTMicroclimate-03,CH1 rooftop,1579.837382


In [49]:
# Add the matched noise sensor to pedestrian data

pedestrian_matched = pedestrian_join.merge(
    sensor_matches[
        ["location_id", "device_id",
         "noise_location", "distance_m"]
    ],
    on="location_id",
    how="left"
)

print("Rows after sensor matching:", len(pedestrian_matched))
pedestrian_matched.head()

Rows after sensor matching: 81366


,location_id,sensing_date,hourday,pedestriancount,sensor_name,latitude,longitude,device_id,noise_location,distance_m
0,1,2026-06-30,9,238,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171
1,1,2026-06-30,10,726,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171
2,1,2026-06-30,11,1162,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171
3,1,2026-06-30,12,2140,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171
4,1,2026-06-30,13,1596,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171


In [50]:
# Join pedestrian and hourly noise data

epic4_data = pedestrian_matched.merge(
    noise_hourly[
        ["device_id", "sensing_date",
         "hourday", "average_noise"]
    ],
    on=["device_id", "sensing_date", "hourday"],
    how="left"
)

print("Final Epic 4 rows:", len(epic4_data))
epic4_data.head()

Final Epic 4 rows: 81366


,location_id,sensing_date,hourday,pedestriancount,sensor_name,latitude,longitude,device_id,noise_location,distance_m,average_noise
0,1,2026-06-30,9,238,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.975
1,1,2026-06-30,10,726,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.975
2,1,2026-06-30,11,1162,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.925
3,1,2026-06-30,12,2140,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.850
4,1,2026-06-30,13,1596,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.875


In [51]:
# Check missing noise matches

print("Missing noise values:",
      epic4_data["average_noise"].isna().sum())

print("Total rows:", len(epic4_data))

print(
    "Noise coverage:",
    epic4_data["average_noise"].notna().mean()
)

epic4_data.head()

Missing noise values: 23463
Total rows: 81366
Noise coverage: 0.7116363100066366


,location_id,sensing_date,hourday,pedestriancount,sensor_name,latitude,longitude,device_id,noise_location,distance_m,average_noise
0,1,2026-06-30,9,238,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.975
1,1,2026-06-30,10,726,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.975
2,1,2026-06-30,11,1162,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.925
3,1,2026-06-30,12,2140,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.850
4,1,2026-06-30,13,1596,Bou292_T,-37.813494,144.965153,ICTMicroclimate-03,CH1 rooftop,196.251171,62.875


In [52]:
# Prepare noise data for backend integration

noise_final = noise_clean.copy()

# Create date and hour columns
noise_final["sensing_date"] = noise_final["received_at"].dt.date
noise_final["hourday"] = noise_final["received_at"].dt.hour

# Keep required columns
noise_final = noise_final[
    [
        "device_id",
        "sensorlocation",
        "sensing_date",
        "hourday",
        "noise",
        "latitude",
        "longitude"
    ]
]

noise_final.head()

,device_id,sensorlocation,sensing_date,hourday,noise,latitude,longitude
0,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,49.099998,-37.818593,144.97164
1,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,49.099998,-37.818593,144.97164
2,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,47.500000,-37.818593,144.97164
3,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,47.900002,-37.818593,144.97164
4,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,1,49.400002,-37.818593,144.97164


In [53]:
# Aggregate noise data by hour

noise_hourly = noise_final.groupby(
    [
        "device_id",
        "sensorlocation",
        "sensing_date",
        "hourday",
        "latitude",
        "longitude"
    ],
    as_index=False
)["noise"].mean()

# Rename the noise column
noise_hourly = noise_hourly.rename(
    columns={"noise": "average_noise"}
)

noise_hourly.head()

,device_id,sensorlocation,sensing_date,hourday,latitude,longitude,average_noise
0,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,-37.818593,144.97164,48.399999
1,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,1,-37.818593,144.97164,47.875000
2,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,2,-37.818593,144.97164,50.274999
3,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,3,-37.818593,144.97164,47.575000
4,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,4,-37.818593,144.97164,53.300002


In [54]:
# Check the final noise dataset

print("Shape:", noise_hourly.shape)

print("\nMissing values:")
print(noise_hourly.isnull().sum())

print("\nDuplicates:")
print(noise_hourly.duplicated().sum())

noise_hourly.head()

Shape: (15519, 7)

Missing values:
device_id         0
sensorlocation    0
sensing_date      0
hourday           0
latitude          0
longitude         0
average_noise     0
dtype: int64

Duplicates:
0


,device_id,sensorlocation,sensing_date,hourday,latitude,longitude,average_noise
0,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,-37.818593,144.97164,48.399999
1,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,1,-37.818593,144.97164,47.875000
2,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,2,-37.818593,144.97164,50.274999
3,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,3,-37.818593,144.97164,47.575000
4,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,4,-37.818593,144.97164,53.300002


In [55]:
# Export cleaned hourly noise data

noise_hourly.to_csv(
    "noise_hourly_clean.csv",
    index=False
)

print("Noise hourly data exported successfully.")

Noise hourly data exported successfully.


In [56]:
# Check the unique key for hourly noise data

noise_key_duplicates = noise_hourly.duplicated(
    subset=["device_id", "sensing_date", "hourday"]
).sum()

print("Duplicate noise keys:", noise_key_duplicates)

Duplicate noise keys: 0


In [57]:
# Check coordinate ranges

print("Latitude range:")
print(noise_hourly["latitude"].min(),
      noise_hourly["latitude"].max())

print("\nLongitude range:")
print(noise_hourly["longitude"].min(),
      noise_hourly["longitude"].max())

Latitude range:
-37.8223306 -37.8128595

Longitude range:
144.9521696 144.9829409


In [58]:
# Check whether each noise device has one sensor location

device_locations = noise_hourly.groupby(
    "device_id"
)[["latitude", "longitude"]].nunique()

multiple_locations = device_locations[
    (device_locations["latitude"] > 1) |
    (device_locations["longitude"] > 1)
]

print("Devices with multiple coordinates:",
      len(multiple_locations))

multiple_locations.head()

Devices with multiple coordinates: 0


,latitude,longitude
device_id,,


In [59]:
# Check final data types

print(noise_hourly[
    [
        "device_id",
        "sensing_date",
        "hourday",
        "latitude",
        "longitude",
        "average_noise"
    ]
].dtypes)

device_id        string[python]
sensing_date             object
hourday                   int32
latitude                float64
longitude               float64
average_noise           float64
dtype: object


In [60]:
# Set final column order for backend integration

noise_hourly = noise_hourly[
    [
        "device_id",
        "sensorlocation",
        "sensing_date",
        "hourday",
        "latitude",
        "longitude",
        "average_noise"
    ]
]

noise_hourly.head()

,device_id,sensorlocation,sensing_date,hourday,latitude,longitude,average_noise
0,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,-37.818593,144.97164,48.399999
1,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,1,-37.818593,144.97164,47.875000
2,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,2,-37.818593,144.97164,50.274999
3,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,3,-37.818593,144.97164,47.575000
4,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,4,-37.818593,144.97164,53.300002


In [61]:
# Final check before export

print("Final shape:", noise_hourly.shape)

print("\nMissing values:")
print(noise_hourly.isnull().sum())

print("\nDuplicate keys:")
print(
    noise_hourly.duplicated(
        subset=["device_id", "sensing_date", "hourday"]
    ).sum()
)

noise_hourly.head()

Final shape: (15519, 7)

Missing values:
device_id         0
sensorlocation    0
sensing_date      0
hourday           0
latitude          0
longitude         0
average_noise     0
dtype: int64

Duplicate keys:
0


,device_id,sensorlocation,sensing_date,hourday,latitude,longitude,average_noise
0,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,0,-37.818593,144.97164,48.399999
1,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,1,-37.818593,144.97164,47.875000
2,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,2,-37.818593,144.97164,50.274999
3,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,3,-37.818593,144.97164,47.575000
4,ICTMicroclimate-01,Birrarung Marr Park - Pole 1131,2026-06-16,4,-37.818593,144.97164,53.300002


In [62]:
# Export join-ready hourly noise data

noise_hourly.to_csv(
    "noise_hourly_clean.csv",
    index=False
)

print("Noise hourly data is ready for backend integration.")

Noise hourly data is ready for backend integration.
